# Semana 05: Estratégias de Testes Automatizados, Pirâmide de Testes e Quality Gates no CI

## Módulo de Automação de Qualidade e Integração Contínua

Nesta aula, abordaremos os fundamentos e a aplicação prática de **Estratégias de Testes Automatizados** em pipelines de Integração Contínua (CI). Analisaremos a estrutura da **Pirâmide de Testes**, a distinção entre **Testes Unitários, de Integração e Ponta a Ponta (E2E)**, a utilização de **Duplos de Teste (Mocks e Stubs)**, a mensuração de **Cobertura de Código (Code Coverage)** e a definição de **Quality Gates** para controle automatizado de qualidade em Pull Requests.

### Objetivos de Aprendizagem
- Compreender os princípios e a distribuição de esforços da **Pirâmide de Testes** (Mike Cohn / Martin Fowler).
- Identificar as características, vantagens e custos de **Testes Unitários**, **Testes de Integração** e **Testes End-to-End (E2E)**.
- Reconhecer os principais anti-padrões de teste na indústria (*Cone de Sorvete* e *Ampulheta*).
- Compreender o papel de **Mocks, Stubs e Fixtures** no isolamento de componentes e eliminação de dependências externas (I/O, banco de dados, rede).
- Interpretar relatórios de **Cobertura de Código (Statement e Branch Coverage)** gerados pelo `pytest-cov`.
- Implementar e configurar **Quality Gates** para validação automatizada de código em pipelines de CI.

---


## 1. Fundamentação Teórica

### 1.1 A Pirâmide de Testes (Testing Pyramid)

A **Pirâmide de Testes** é um modelo conceitual concebido por Mike Cohn e detalhado por Martin Fowler. Ela orienta a distribuição ideal dos diferentes tipos de testes automatizados em um projeto de software, equilibrando **velocidade de execução**, **custo de desenvolvimento/manutenção** e **confiabilidade**.

![Pirâmide de Testes](img/piramide_de_testes.jpg)

### Comparativo das Camadas da Pirâmide

| Camada | Proporção Recomendada | Escopo de Validação | Velocidade de Execução | Custo e Fragilidade | Isolamento de Dependências |
| :--- | :---: | :--- | :---: | :---: | :---: |
| **Testes Unitários (Base)** | ~70% | Funções, classes e métodos isolados | Milissegundos | Baixo (Muito estáveis) | Total (Uso de Mocks/Stubs) |
| **Testes de Integração (Meio)** | ~20% | Interação entre múltiplos módulos/APIs/DB | Segundos | Médio | Parcial (Pode usar DB em memória) |
| **Testes de Ponta a Ponta / E2E (Topo)** | ~10% | Fluxo completo do usuário (UI + Backend + DB real) | Minutos | Alto (Suscetíveis a *Flaky Tests*) | Nenhum (Ambiente real integrado) |

---


### 1.2 Detalhamento de Cada Nível de Teste

#### 1. Testes Unitários (Unit Tests)
- **Foco:** Validar se a menor unidade de código testável (uma função matemática, uma validação de regra de negócio, um método de cálculo) produz a saída esperada para entradas específicas.
- **Princípio Fundamental:** Isolamento estrito. Testes unitários **não** devem fazer chamadas de rede, conexões de banco de dados nem leitura de arquivos em disco.
- **Vantagens:** Execução quase instantânea; fornecem diagnóstico preciso do ponto exato da falha; viabilizam práticas como TDD (Test-Driven Development).

#### 2. Testes de Integração (Integration Tests)
- **Foco:** Validar se a comunicação e a troca de mensagens entre componentes distintos funcionam de acordo com o contrato esperado.
- **Exemplos:** Validação de rotas HTTP em controladores/rotas Flask, persistência real de dados em um banco relacional, serialização de mensagens JSON enviadas a um broker MQTT/RabbitMQ.
- **Vantagens:** Detectam erros de integração de protocolos, mapeamento objeto-relacional (ORM) e contratos de API que os testes unitários isolados não conseguem capturar.

#### 3. Testes de Ponta a Ponta (End-to-End / E2E)
- **Foco:** Simular a jornada completa do usuário final a partir da camada mais externa (interface web, mobile ou chamadas de API ponta a ponta) transitando por toda a infraestrutura até o banco de dados final.
- **Desvantagens e Desafios:** Tempo elevado de execução (minutos ou horas); alto custo de infraestrutura de teste; dependência de navegadores ou emuladores; propensão a *Flaky Tests* (testes intermitentes que falham por atrasos de rede ou renderização, sem defeito real no código).

---


### 1.3 Anti-Padrões da Pirâmide de Testes

Quando as equipes não seguem os princípios da pirâmide de testes, surgem dois padrões disfuncionais comuns na engenharia de software:

#### 1. O Cone de Sorvete (Ice Cream Cone / Inverted Pyramid)
Ocorre quando a organização possui uma base ínfima de testes unitários, quase nenhum teste de integração e uma camada massiva de testes manuais ou E2E demorados.
- **Consequência:** Pipelines de CI extremamente lentos, feedback tardio para os desenvolvedores, alto custo de manutenção e bugs complexos descobertos apenas em produção.

#### 2. A Ampulheta (Hourglass)
Ocorre quando o projeto possui muitos testes unitários e muitos testes E2E, mas negligencia completamente os testes de integração.
- **Consequência:** Erros de comunicação entre microsserviços e contratos de banco de dados só são descobertos na camada E2E, gerando alto custo para depuração e identificação da causa raiz.

---


### 1.4 Duplos de Teste: Mocks, Stubs e Fixtures

Para garantir que os testes unitários sejam rápidos, determinísticos e isolados, utilizamos **Duplos de Teste (Test Doubles)**:

- **Stub:** Objeto simplificado que retorna respostas pré-programadas para chamadas específicas durante o teste (ex: um leitor de sensor que sempre retorna `temperatura = 25.0`).
- **Mock:** Objeto programável que, além de fornecer respostas simuladas, registra se foi chamado, quantas vezes e com quais parâmetros específicos (verificação de comportamento).
- **Fixture (Pytest):** Função reutilizável que prepara o ambiente (contexto inicial, instâncias de serviço, dados de teste) antes da execução de um teste e realiza a limpeza posterior.

---


### 1.5 Cobertura de Código e Quality Gates no CI

#### Cobertura de Código (Code Coverage)
A cobertura de código mensura o percentual de linhas executadas durante os testes automatizados:
- **Statement Coverage:** Percentual de linhas executadas.
- **Branch Coverage:** Percentual de ramificações condicionais (`if/else`) testadas.

> **Nota Importante:** Alta cobertura de código é uma condição necessária, mas não suficiente para garantir qualidade. Um código com 100% de cobertura ainda pode conter falhas se as asserções (`assert`) não validarem os casos de borda e regras de negócio essenciais.

#### O Conceito de Quality Gate
Um **Quality Gate** é uma barreira de controle automatizada configurada no pipeline de CI/CD. Se o código submetido em um Pull Request não atender aos critérios mínimos estabelecidos, o pipeline falha e o merge é bloqueado.

```text
  [Pull Request Aberto]
           |
           v
  [Execucao do Pipeline de CI]
  (Pytest + Pytest-Cov + Linter)
           |
           v
  +--------------------------------+
  |      CRITERIOS DO GATE:        |
  |  - Testes com falha == 0       |
  |  - Cobertura >= 80%            |
  |  - Erros de Linting == 0       |
  +--------------------------------+
          /                \
       (APROVADO)      (REPROVADO)
         /                  \
        v                    v
  [Merge Permitido]   [Build Bloqueado / PR Travado]
```

---


## 2. Prática em Python: Suíte de Testes e Simulador de Quality Gate

Abaixo, implementamos um sistema de controle de produção industrial contendo:
1. **Classe de Domínio (`OrdemProducao`):** Validação de regras e cálculos.
2. **Camada de Serviço (`ControleProducaoService`):** Lógica integrada com repositório.
3. **Suíte de Testes Unitários:** Validação isolada de regras de domínio.
4. **Suíte de Testes de Integração:** Validação da camada de serviço com Mock de repositório.
5. **Motor de Quality Gate:** Validador automatizado que avalia aprovação de Pull Request.


In [ ]:
# 1. MODELO DE DOMINIO (Entidade de Negocio)
class OrdemProducao:
    def __init__(self, id_ordem: str, pecas_planejadas: int, pecas_produzidas: int = 0, pecas_defeituosas: int = 0):
        if pecas_planejadas <= 0:
            raise ValueError("O volume planejado deve ser maior que zero.")
        if pecas_produzidas < 0 or pecas_defeituosas < 0:
            raise ValueError("Quantidades de pecas nao podem ser negativas.")
        if pecas_defeituosas > pecas_produzidas:
            raise ValueError("Pecas defeituosas nao podem exceder o total produzido.")

        self.id_ordem = id_ordem
        self.pecas_planejadas = pecas_planejadas
        self.pecas_produzidas = pecas_produzidas
        self.pecas_defeituosas = pecas_defeituosas

    def pecas_conformes(self) -> int:
        return self.pecas_produzidas - self.pecas_defeituosas

    def taxa_qualidade(self) -> float:
        if self.pecas_produzidas == 0:
            return 0.0
        return round((self.pecas_conformes() / self.pecas_produzidas) * 100.0, 2)

    def eficiencia_producao(self) -> float:
        return round((self.pecas_conformes() / self.pecas_planejadas) * 100.0, 2)


# 2. CAMADA DE SERVICO (Logica Integrada com Repositorio)
class RepositorioOrdensMock:
    """Mock que simula a persistencia em banco de dados."""
    def __init__(self):
        self.banco = {}

    def salvar(self, ordem: OrdemProducao):
        self.banco[ordem.id_ordem] = ordem

    def buscar_por_id(self, id_ordem: str):
        return self.banco.get(id_ordem)


class ControleProducaoService:
    def __init__(self, repositorio):
        self.repositorio = repositorio

    def registrar_ordem(self, id_ordem: str, pecas_planejadas: int) -> OrdemProducao:
        ordem = OrdemProducao(id_ordem, pecas_planejadas)
        self.repositorio.salvar(ordem)
        return ordem

    def atualizar_apontamento(self, id_ordem: str, produzidas: int, defeituosas: int) -> OrdemProducao:
        ordem = self.repositorio.buscar_por_id(id_ordem)
        if not ordem:
            raise KeyError(f"Ordem {id_ordem} nao encontrada.")
        
        ordem_atualizada = OrdemProducao(id_ordem, ordem.pecas_planejadas, produzidas, defeituosas)
        self.repositorio.salvar(ordem_atualizada)
        return ordem_atualizada


In [ ]:
# 3. SUITES DE TESTES E MOTOR DO QUALITY GATE
def executar_testes_unitarios() -> list:
    resultados = []
    
    # Teste Unitario 1: Calculo de pecas conformes e taxa de qualidade
    try:
        op = OrdemProducao(id_ordem="OP-101", pecas_planejadas=1000, pecas_produzidas=800, pecas_defeituosas=40)
        assert op.pecas_conformes() == 760
        assert op.taxa_qualidade() == 95.0
        assert op.eficiencia_producao() == 76.0
        resultados.append(("test_unit_calculos_ordem_valida", True, "OK"))
    except Exception as e:
        resultados.append(("test_unit_calculos_ordem_valida", False, str(e)))

    # Teste Unitario 2: Validacao de volume planejado invalido (<= 0)
    try:
        try:
            OrdemProducao(id_ordem="OP-102", pecas_planejadas=0)
            resultados.append(("test_unit_planejado_zero_lanca_erro", False, "Esperava ValueError"))
        except ValueError:
            resultados.append(("test_unit_planejado_zero_lanca_erro", True, "OK"))
    except Exception as e:
        resultados.append(("test_unit_planejado_zero_lanca_erro", False, str(e)))

    # Teste Unitario 3: Defeitos maiores que producao lanca excecao
    try:
        try:
            OrdemProducao(id_ordem="OP-103", pecas_planejadas=500, pecas_produzidas=100, pecas_defeituosas=120)
            resultados.append(("test_unit_defeitos_maior_que_produzido", False, "Esperava ValueError"))
        except ValueError:
            resultados.append(("test_unit_defeitos_maior_que_produzido", True, "OK"))
    except Exception as e:
        resultados.append(("test_unit_defeitos_maior_que_produzido", False, str(e)))

    return resultados


def executar_testes_integracao() -> list:
    resultados = []
    repo_mock = RepositorioOrdensMock()
    service = ControleProducaoService(repo_mock)

    # Teste de Integracao 1: Criacao e persistencia de ordem via servico
    try:
        op = service.registrar_ordem("OP-200", 500)
        op_recuperada = repo_mock.buscar_por_id("OP-200")
        assert op_recuperada is not None
        assert op_recuperada.pecas_planejadas == 500
        resultados.append(("test_integration_registro_ordem", True, "OK"))
    except Exception as e:
        resultados.append(("test_integration_registro_ordem", False, str(e)))

    # Teste de Integracao 2: Atualizacao de apontamento e recalculo integrado
    try:
        op_att = service.atualizar_apontamento("OP-200", produzidas=400, defeituosas=20)
        assert op_att.taxa_qualidade() == 95.0
        assert op_att.eficiencia_producao() == 76.0
        resultados.append(("test_integration_atualizacao_apontamento", True, "OK"))
    except Exception as e:
        resultados.append(("test_integration_atualizacao_apontamento", False, str(e)))

    return resultados


def avaliar_quality_gate(resultados_unit, resultados_integracao, cobertura_obtida_pct: float, meta_minima_cov: float = 80.0):
    todos_testes = resultados_unit + resultados_integracao
    total = len(todos_testes)
    passados = sum(1 for _, ok, _ in todos_testes if ok)
    falhas = total - passados
    
    aprovado = (falhas == 0) and (cobertura_obtida_pct >= meta_minima_cov)
    
    print("=" * 65)
    print("RELATORIO DE EXECUCAO DO PIPELINE DE TESTES E QUALITY GATE")
    print("=" * 65)
    print(f"Testes Unitarios Executados:   {len(resultados_unit)}")
    for nome, ok, msg in resultados_unit:
        status = "PASSED" if ok else f"FAILED ({msg})"
        print(f"  - [UNIT] {nome}: {status}")
        
    print(f"\nTestes de Integracao Executados: {len(resultados_integracao)}")
    for nome, ok, msg in resultados_integracao:
        status = "PASSED" if ok else f"FAILED ({msg})"
        print(f"  - [INT]  {nome}: {status}")
        
    print("-" * 65)
    print(f"Total de Testes:              {total}")
    print(f"Testes com Sucesso:           {passados}")
    print(f"Testes com Falha:             {falhas}")
    print(f"Cobertura de Codigo Obtida:   {cobertura_obtida_pct:.1f}%")
    print(f"Meta Minima do Quality Gate:  {meta_minima_cov:.1f}%")
    print("-" * 65)
    
    if aprovado:
        print("RESULTADO DO QUALITY GATE: APROVADO")
        print("Status do Pipeline: SUCESSO - Merge do Pull Request autorizado.")
    else:
        print("RESULTADO DO QUALITY GATE: REPROVADO")
        print("Status do Pipeline: FALHA - Merge bloqueado automaticamente no GitHub.")
    print("=" * 65)


# Execucao do Pipeline de Testes
res_unit = executar_testes_unitarios()
res_int = executar_testes_integracao()
avaliar_quality_gate(res_unit, res_int, cobertura_obtida_pct=91.4, meta_minima_cov=80.0)


---

## 3. Exercícios de Fixação e Avaliação

### Questão 1 (Pirâmide de Testes)
De acordo com o modelo da Pirâmide de Testes de Mike Cohn e Martin Fowler, explique por que a base da pirâmide deve ser composta majoritariamente por **Testes Unitários** (~70%) em vez de **Testes de Ponta a Ponta (E2E)**. Aborde em sua resposta os fatores de velocidade de execução, custo de manutenção e isolamento de falhas.

### Questão 2 (Anti-Padrões de Testes)
Explique o que é o anti-padrão do **Cone de Sorvete (Ice Cream Cone)** em estratégias de testes automatizados. Quais são os impactos operacionais imediatos no pipeline de Integração Contínua (CI) e na produtividade do time de desenvolvimento quando um projeto opera sob esse anti-padrão?

### Questão 3 (Duplos de Teste / Mocks)
Qual é a finalidade do uso de **Mocks** e **Stubs** em testes unitários? Por que um teste unitário de uma aplicação nunca deve se conectar a um banco de dados real em rede ou a um broker MQTT externo durante a execução no pipeline de CI?

### Questão 4 (Quality Gates no GitHub Actions)
Como você configuraria o comando de teste no arquivo de workflow `.github/workflows/ci.yml` para garantir que o job de CI falhe automaticamente caso a cobertura total de código do projeto seja inferior a **85%** utilizando o `pytest-cov`?
